# syft-bg Python API Test

Tests the new Pythonic API for syft-bg:
1. Install from PR branch
2. `syft_bg.init(email=..., start=True)`
3. `syft_bg.status`
4. `syft_bg.start()` / `stop()` / `restart()` / `ensure_running()`
5. `syft_bg.auto_approve()`
6. `syft_bg.authenticate()` (interactive, optional)
7. `syft_bg.logs()`

In [ ]:
# Cell 1: Install from PR branch
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

BRANCH = "koen/syft-bg-docs"
REPO = "https://github.com/OpenMined/syft-client.git"

# Install syft-client from branch (has daemon Colab fix)
!pip install -q "syft-client @ git+{REPO}@{BRANCH}"
# Override syft-bg and syft-job with branch versions
!pip install -q --force-reinstall --no-deps "syft-bg @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-bg"
!pip install -q --force-reinstall --no-deps "syft-job @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-job"

In [ ]:
# Cell 2: Verify import and version
import syft_bg

print(f"syft_bg version: {syft_bg.__version__}")
print(f"Available API: {[x for x in syft_bg.__all__]}")

In [ ]:
# Cell 3: Check status before init (should show 'not configured')
syft_bg.status

In [ ]:
# Cell 4: Init without start (just creates config)
# Replace with your email
EMAIL = "YOUR_EMAIL@example.com"

result = syft_bg.init(email=EMAIL, skip_oauth=True)
result

In [ ]:
# Cell 5: Check status after init
syft_bg.status

In [ ]:
# Cell 6: Init with start=True (will fail if tokens missing — that's expected)
result = syft_bg.init(email=EMAIL, start=True, skip_oauth=True)
result

In [ ]:
# Cell 7: Authenticate (interactive — sets up Gmail + Drive tokens)
# Skip this cell if tokens already exist
# syft_bg.authenticate()

In [ ]:
# Cell 8: Start services manually
syft_bg.start()

In [ ]:
# Cell 9: Check status — services should be running
syft_bg.status

In [ ]:
# Cell 10: Stop and restart
print("--- stop ---")
print(syft_bg.stop())

print("\n--- restart ---")
print(syft_bg.restart())

print("\n--- ensure_running ---")
print(syft_bg.ensure_running())

In [ ]:
# Cell 11: Create a test script and auto-approve it
from pathlib import Path
import tempfile

script_dir = Path(tempfile.mkdtemp(prefix="syft_bg_test_"))
(script_dir / "main.py").write_text('print("hello from auto-approved script")')

result = syft_bg.auto_approve(
    contents=[str(script_dir / "main.py")],
    file_names=["params.json"],
    peers=["alice@example.com"],
)
result

In [ ]:
# Cell 12: Create another auto-approval with multiple scripts
(script_dir / "utils.py").write_text('def helper(): return 42')

result2 = syft_bg.auto_approve(
    contents=[str(script_dir / "main.py"), str(script_dir / "utils.py")],
    name="multi_script_analysis",
    peers=["bob@example.com", "charlie@example.com"],
)
result2

In [ ]:
# Cell 13: Auto-approve with no peer restriction (any peer)
result3 = syft_bg.auto_approve(
    contents=[str(script_dir / "main.py")],
    name="open_approval",
)
result3

In [ ]:
# Cell 14: Check status — should show all auto-approval objects
syft_bg.status

In [ ]:
# Cell 15: View logs
print("--- approve logs ---")
for line in syft_bg.logs("approve", n=10):
    print(line)

print("\n--- notify logs ---")
for line in syft_bg.logs("notify", n=10):
    print(line)

In [ ]:
# Cell 16: Stop all services
syft_bg.stop()

In [ ]:
# Cell 17: Final status check
syft_bg.status